In [0]:
%python
# ---
# PROJECT: NYC Ride Analytics - Data Engineering Pipeline
# STAGE: Bronze Layer (Raw Ingestion)
# OBJECTIVE: Securely connecting to Azure Data Lake and ingesting raw taxi data
# ---

# --- SECURITY WARNING ---
# Sensitive credentials have been redacted for GitHub upload. 
# In a production "Pod," these are retrieved via Databricks Secrets:
# client_secret = dbutils.secrets.get(scope="analytics-scope", key="adls-secret")
# ------------------------

# Service principal credentials for Azure Entra ID authentication
client_id = "REDACTED_CLIENT_ID"
tenant_id = "REDACTED_TENANT_ID"
client_secret = "REDACTED_CLIENT_SECRET"
storage_account_name = "datalakerides" 

# Setting Spark configurations to authenticate with ADLS Gen2 directly.
# This method uses OAuth2 and avoids DBFS mounting, which is the 
# modern standard for secure, high-concurrency Databricks environments.
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("Status: Spark session authentication logic configured (Credentials Redacted).")

Status: Spark session authenticated with Azure Storage.


In [0]:
# ---
# PRE-PROCESSING: Ensure the destination folder is clean for a fresh Delta write
# SOURCE: Public Databricks NYC Taxi Sample Dataset
# ---

# Define the landing zone path in our 'raw-data' container
target_path = f"abfss://raw-data@{storage_account_name}.dfs.core.windows.net/ride_data_raw"

# Clean up existing files in the target directory to prevent schema/format collisions
print(f"Action: Removing existing data at {target_path}...")
dbutils.fs.rm(target_path, recurse=True)

# Define the source path for the public NYC taxi data
source_path = "/databricks-datasets/nyctaxi/tables/nyctaxi_yellow"

Action: Removing existing data at abfss://raw-data@datalakerides.dfs.core.windows.net/ride_data_raw...


In [0]:
# ---
# INGESTION: Transferring data from the public source to our private lake
# We are switching the reader to 'delta' format to match the source's transaction log.
# ---

print("Action: Initiating data transfer to Bronze layer...")

# Load the raw data—updated to format("delta") to fix the Incompatible Format error
df_raw = spark.read.format("delta").load(source_path).limit(50000)

# Write the data into our storage account. 
# We keep this as 'delta' to build our own transaction history.
df_raw.write.format("delta").mode("overwrite").save(target_path)

print(f"🎉 SUCCESS: Ingestion complete. Raw data landed at: {target_path}")

Action: Initiating data transfer to Bronze layer...
🎉 SUCCESS: Ingestion complete. Raw data landed at: abfss://raw-data@datalakerides.dfs.core.windows.net/ride_data_raw
